# Suite PHY — The physical layout

`Physical` defines the coordinates physical layout gives the logical layout's points, the container layout's frames and a diagram's viewport, each an annotation owned by what it annotates. mbse-basis [CAP-241: physical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-241); its components, which place them, are TypeScript's only.

In [ ]:
import { assert, Bench, text } from "./support.js";
import { JSON as Json, Paths, Validators } from "@mbse/schemas/Framework";
import { Container, Logical, Physical } from "../src/Framework/index.js";

const b = new Bench();
b.store.transaction(Physical.install);
const [L, P] = [b.store.Logical, b.store.Physical];

function at(x: number, y: number): (v: any) => any {
  return (v: any) => v.x(x).y(y);
}

function size(width: number, height: number): (v: any) => any {
  return (v: any) => v.width(width).height(height);
}

<a id="PHY-01"></a>
## PHY-01 · The install, after Logical's and Container's, defines the physical layout's schemas, each valid, and run again writes nothing

In [ ]:
const NAMES = ["Physical.OfPoint2", "Physical.OfSize", "Physical.Placement", "Physical.OfPlacement", "Physical.GridFrame", "Physical.CellFrame", "Physical.OfFrame", "Physical.Viewport", "Physical.OfViewport"];
const ANNOTATED: Record<string, string> = { "Logical.OfPoint": "placement", "Container.OfGrid": "frame",
  "Container.OfCell": "frame", "Logical.OfDiagram": "viewport" };
const schemas = NAMES.map((name) => b.store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
b.store.transaction(Physical.install);
assert(NAMES.every((name, i) => b.store.registered(name) === schemas[i]));
assert(Object.entries(ANNOTATED).every(([name, member]) => b.store.registered(name).adjacencies.has(member)));

<a id="PHY-02"></a>
## PHY-02 · A point's placement is owned by the point: its absolute and relative coordinates and its direction, and its path the point's

In [ ]:
assert(b.grid("cpu", 2n) === "[{\"kind\":\"grid\",\"shape\":\"cpu\",\"span\":2,\"grows\":\"rows\",\"cell_rotation\":0}]");
assert(b.child("cpu", "a") === "[{\"kind\":\"place\",\"shape\":\"a\",\"order\":0,\"to\":[0,0]},{\"kind\":\"extents\",\"rows\":1,\"columns\":1}]");
const a = b.shape("cpu.a");
b.store.transaction(() => L.OfShape(a, (s: any) => s.origin((x: any) => x.insert((e: any) => e.point((p: any) => p)))));
const [o] = [...a.origin].map((e: any) => e.point);
b.store.transaction(() => L.OfPoint(o, (p: any) => p.placement((x: any) => x.insert((e: any) => e.placement(
  (q: any) => q.absolute(at(40, 30)).relative(at(40, 30)).direction(0))))));
const [placement] = [...o.placement].map((e: any) => e.placement);
const path = (x: any): string => Json.ToJSON(b.store)(b.store.Paths.OfRoot.as_schema(), Paths.of(x));
assert(text([placement.absolute.x.get(), placement.absolute.y.get(), placement.relative.x.get(), placement.direction.get()]) === "[40,30,40,0]");
assert(text([...placement.point].map((e: any) => e.point === o).join() === "true") === "true");
assert(text(path(placement).slice(path(o).length - 13)) === "\"{\\\"i\\\": \\\"placement\\\", \\\"o\\\": \\\"point\\\", \\\"properties\\\": {}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}}}\"");
assert(text(Json.ToJSON(b.store)(P.OfPlacement.as_schema(), placement)) === "\"{\\\"mutations\\\": [{\\\"upsert\\\": {\\\"exists\\\": [{\\\"symbols\\\": {\\\"0\\\": {\\\"schema\\\": [{\\\"$ref\\\": {\\\"singleton\\\": \\\".\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"schemas\\\", \\\"o\\\": \\\"\\\", \\\"properties\\\": {\\\"name\\\": \\\"Physical\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"schemas\\\", \\\"o\\\": \\\"\\\", \\\"properties\\\": {\\\"name\\\": \\\"OfPlacement\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}]}}, \\\"spec\\\": [{\\\"0\\\": [{\\\"as_property\\\": {\\\"name\\\": \\\"absolute\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": {\\\"x\\\": 40.0, \\\"y\\\": 30.0}}}]}}, {\\\"as_property\\\": {\\\"name\\\": \\\"relative\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": {\\\"x\\\": 40.0, \\\"y\\\": 30.0}}}]}}, {\\\"as_property\\\": {\\\"name\\\": \\\"direction\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"value\\\": 0.0}}]}}, {\\\"as_adjacency\\\": {\\\"name\\\": \\\"point\\\", \\\"mutations\\\": [{\\\"insert\\\": {\\\"entries\\\": [{\\\"mutations\\\": [{\\\"as_link\\\": {\\\"name\\\": \\\"point\\\", \\\"mutations\\\": [{\\\"set\\\": {\\\"ref\\\": {\\\"path\\\": {\\\"singleton\\\": \\\"Logical.Diagrams\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"diagrams\\\", \\\"o\\\": \\\"directory\\\", \\\"properties\\\": {\\\"name\\\": \\\"overview\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"shapes\\\", \\\"o\\\": \\\"diagram\\\", \\\"properties\\\": {\\\"name\\\": \\\"cpu\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"children\\\", \\\"o\\\": \\\"container\\\", \\\"properties\\\": {\\\"name\\\": \\\"a\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"origin\\\", \\\"o\\\": \\\"origin_of\\\", \\\"properties\\\": {}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}}}}]}}]}]}}]}}]}]}]}}]}\"");

<a id="PHY-03"></a>
## PHY-03 · A grid's and a cell's frame, and a diagram's viewport, each owned by what it annotates

In [ ]:
const [entry] = [...b.shape("cpu").grid];
const grid = entry.grid;
const [cell] = [...grid.cells].map((e: any) => e.cell);
b.store.transaction(() => {
  b.store.Container.OfGrid(grid, (g: any) => g.frame((x: any) => x.insert((e: any) => e.frame(
    (f: any) => f.absolute(at(0, 0)).relative(at(0, 0)).size(size(200, 80))))));
  b.store.Container.OfCell(cell, (c: any) => c.frame((x: any) => x.insert((e: any) => e.frame(
    (f: any) => f.absolute(at(0, 0)).relative(at(0, 0)).size(size(100, 80))))));
  L.OfDiagram(b.overview, (d: any) => d.viewport((x: any) => x.insert((e: any) => e.viewport(
    (v: any) => v.size(size(800, 600))))));
});
const [frame] = [...cell.frame].map((e: any) => e.frame);
assert(text([[...grid.frame].map((e: any) => e.frame.size.width.get()), frame.size.width.get(), [...b.overview.viewport].map((e: any) => e.viewport.size.height.get())]) === "[[200],100,[600]]");
assert(text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) === "[]");

<a id="PHY-04"></a>
## PHY-04 · One placement per point, as the validator reports

In [ ]:
b.store.transaction(() => L.OfPoint(o, (p: any) => p.placement((x: any) => x.insert((e: any) => e.placement(
  (q: any) => q.direction(90))))));
assert(text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) === "[\"unique(placement) violated: entries agreeing on ['point'] differ on ['placement']\"]");

<a id="PHY-05"></a>
## PHY-05 · An annotation goes with what it annotates: a child removed takes its points' placements, and its cell's frame

In [ ]:
assert(b.unchild("cpu", "a") === "[{\"kind\":\"leave\",\"shape\":\"a\",\"from\":[0,0]},{\"kind\":\"extents\",\"rows\":0,\"columns\":0}]");
assert(text([b.store.holds(placement), b.store.holds(frame), b.store.holds(grid)]) === "[false,false,false]");